In [1]:
import findspark
findspark.init()
findspark.find()
import pyspark
from pyspark import SparkContext
from pyspark.sql import SparkSession, SQLContext

In [2]:
sc=SparkContext()

23/01/22 19:15:58 WARN Utils: Your hostname, LAPTOP-CUMRVF08 resolves to a loopback address: 127.0.1.1; using 172.24.127.61 instead (on interface eth0)
23/01/22 19:15:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


23/01/22 19:16:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark = SparkSession(sc)

In [4]:
df = "../data/country_level_data_0.csv"

Data Exploration

In [5]:
dataset = spark.read.csv(df,inferSchema=True,header =True)

In [38]:
from pyspark.sql.functions import col, count, isnan, when,lit,round
from pyspark.sql.types import DoubleType
from pyspark.sql.types import IntegerType,BooleanType,DateType,StringType

In [7]:
dataset.printSchema()

root
 |-- region_id: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- gdp: string (nullable = true)
 |-- composition_food_organic_waste_percent: string (nullable = true)
 |-- composition_glass_percent: string (nullable = true)
 |-- composition_metal_percent: string (nullable = true)
 |-- composition_other_percent: string (nullable = true)
 |-- composition_paper_cardboard_percent: string (nullable = true)
 |-- composition_plastic_percent: string (nullable = true)
 |-- composition_rubber_leather_percent: string (nullable = true)
 |-- composition_wood_percent: string (nullable = true)
 |-- composition_yard_garden_green_waste_percent: string (nullable = true)
 |-- other_information_information_system_for_solid_waste_management: string (nullable = true)
 |-- other_information_national_agency_to_enforce_solid_waste_laws_and_regulations: string (nullable = true)
 |-- other_information_national_law_governing_solid_waste_management_in_the_country: string (nullable = tru

In [9]:
dataset.head(2)

[Row(region_id='LCN', country_name='Aruba', gdp='35563.3125', composition_food_organic_waste_percent='NA', composition_glass_percent='NA', composition_metal_percent='NA', composition_other_percent='NA', composition_paper_cardboard_percent='NA', composition_plastic_percent='NA', composition_rubber_leather_percent='NA', composition_wood_percent='NA', composition_yard_garden_green_waste_percent='NA', other_information_information_system_for_solid_waste_management='NA', other_information_national_agency_to_enforce_solid_waste_laws_and_regulations='Yes', other_information_national_law_governing_solid_waste_management_in_the_country='Yes', other_information_ppp_rules_and_regulations='Yes', other_information_summary_of_key_solid_waste_information_made_available_to_the_public='Yes', population_population_number_of_people=103187, special_waste_agricultural_waste_tons_year='NA', special_waste_construction_and_demolition_waste_tons_year='NA', special_waste_e_waste_tons_year='NA', special_waste_ha

In [91]:
df1 = dataset
df1.dropDuplicates()

DataFrame[region_id: string, country_name: string, gdp: string, composition_food_organic_waste_percent: string, composition_glass_percent: string, composition_metal_percent: string, composition_other_percent: string, composition_paper_cardboard_percent: string, composition_plastic_percent: string, composition_rubber_leather_percent: string, composition_wood_percent: string, composition_yard_garden_green_waste_percent: string, other_information_information_system_for_solid_waste_management: string, other_information_national_agency_to_enforce_solid_waste_laws_and_regulations: string, other_information_national_law_governing_solid_waste_management_in_the_country: string, other_information_ppp_rules_and_regulations: string, other_information_summary_of_key_solid_waste_information_made_available_to_the_public: string, population_population_number_of_people: int, special_waste_agricultural_waste_tons_year: string, special_waste_construction_and_demolition_waste_tons_year: string, special_wa

In [92]:
df2 = df1.drop(
            'region_id',
            'waste_collection_coverage_rural_percent_of_geographic_area',
            'waste_collection_coverage_rural_percent_of_households',
            'waste_collection_coverage_rural_percent_of_population',
            'waste_collection_coverage_rural_percent_of_waste',
            'waste_collection_coverage_total_percent_of_geographic_area',
            'waste_collection_coverage_total_percent_of_households',
            'waste_collection_coverage_urban_percent_of_geographic_area',
            'waste_collection_coverage_urban_percent_of_households',
            'waste_collection_coverage_urban_percent_of_population',
            'waste_collection_coverage_urban_percent_of_waste',
            'waste_treatment_anaerobic_digestion_percent',
            'waste_treatment_other_percent',
            'waste_treatment_sanitary_landfill_landfill_gas_system_percent',
            'waste_treatment_waterways_marine_percent',
            'where_where_is_this_data_measured',
            'waste_treatment_landfill_unspecified_percent',
            'waste_treatment_incineration_percent',
            'waste_treatment_controlled_landfill_percent',
            'waste_treatment_compost_percent',
            'waste_collection_coverage_total_percent_of_population',
            'waste_collection_coverage_total_percent_of_waste',
            'waste_treatment_open_dump_percent')

In [93]:
df2.head(2)

[Row(country_name='Aruba', gdp='35563.3125', composition_food_organic_waste_percent='NA', composition_glass_percent='NA', composition_metal_percent='NA', composition_other_percent='NA', composition_paper_cardboard_percent='NA', composition_plastic_percent='NA', composition_rubber_leather_percent='NA', composition_wood_percent='NA', composition_yard_garden_green_waste_percent='NA', other_information_information_system_for_solid_waste_management='NA', other_information_national_agency_to_enforce_solid_waste_laws_and_regulations='Yes', other_information_national_law_governing_solid_waste_management_in_the_country='Yes', other_information_ppp_rules_and_regulations='Yes', other_information_summary_of_key_solid_waste_information_made_available_to_the_public='Yes', population_population_number_of_people=103187, special_waste_agricultural_waste_tons_year='NA', special_waste_construction_and_demolition_waste_tons_year='NA', special_waste_e_waste_tons_year='NA', special_waste_hazardous_waste_ton

In [94]:
for col in df2.columns:
    print(col,"with null values: ", df2.filter(df2[col].isNull()).count())

country_name with null values:  0
gdp with null values:  0
composition_food_organic_waste_percent with null values:  0
composition_glass_percent with null values:  0
composition_metal_percent with null values:  0
composition_other_percent with null values:  0
composition_paper_cardboard_percent with null values:  0
composition_plastic_percent with null values:  0
composition_rubber_leather_percent with null values:  0
composition_wood_percent with null values:  0
composition_yard_garden_green_waste_percent with null values:  0
other_information_information_system_for_solid_waste_management with null values:  0
other_information_national_agency_to_enforce_solid_waste_laws_and_regulations with null values:  0
other_information_national_law_governing_solid_waste_management_in_the_country with null values:  0
other_information_ppp_rules_and_regulations with null values:  0
other_information_summary_of_key_solid_waste_information_made_available_to_the_public with null values:  0
population_

In [95]:
for col in df2.columns:
    print(col,"with null values: ", df2.filter(df2[col]=="NA").count())

country_name with null values:  1
gdp with null values:  1
composition_food_organic_waste_percent with null values:  41
composition_glass_percent with null values:  46
composition_metal_percent with null values:  47
composition_other_percent with null values:  42
composition_paper_cardboard_percent with null values:  41
composition_plastic_percent with null values:  42
composition_rubber_leather_percent with null values:  164
composition_wood_percent with null values:  153
composition_yard_garden_green_waste_percent with null values:  179
other_information_information_system_for_solid_waste_management with null values:  105
other_information_national_agency_to_enforce_solid_waste_laws_and_regulations with null values:  53
other_information_national_law_governing_solid_waste_management_in_the_country with null values:  23
other_information_ppp_rules_and_regulations with null values:  55
other_information_summary_of_key_solid_waste_information_made_available_to_the_public with null value

In [96]:
df2.filter(df2["gdp"] == "NA").collect()

[Row(country_name='Sint Maarten (Dutch part)', gdp='NA', composition_food_organic_waste_percent='46', composition_glass_percent='7', composition_metal_percent='7', composition_other_percent='12', composition_paper_cardboard_percent='15', composition_plastic_percent='13', composition_rubber_leather_percent='NA', composition_wood_percent='NA', composition_yard_garden_green_waste_percent='NA', other_information_information_system_for_solid_waste_management='NA', other_information_national_agency_to_enforce_solid_waste_laws_and_regulations='Yes', other_information_national_law_governing_solid_waste_management_in_the_country='Yes', other_information_ppp_rules_and_regulations='NA', other_information_summary_of_key_solid_waste_information_made_available_to_the_public='Yes', population_population_number_of_people=37685, special_waste_agricultural_waste_tons_year='NA', special_waste_construction_and_demolition_waste_tons_year='NA', special_waste_e_waste_tons_year='NA', special_waste_hazardous_w

In [97]:
df2 = df2.replace("NA",None)

In [98]:
for col in df2.columns:
    print(col,"with null values: ", df2.filter(df2[col].isNull()).count())

country_name with null values:  1
gdp with null values:  1
composition_food_organic_waste_percent with null values:  41
composition_glass_percent with null values:  46
composition_metal_percent with null values:  47
composition_other_percent with null values:  42
composition_paper_cardboard_percent with null values:  41
composition_plastic_percent with null values:  42
composition_rubber_leather_percent with null values:  164
composition_wood_percent with null values:  153
composition_yard_garden_green_waste_percent with null values:  179
other_information_information_system_for_solid_waste_management with null values:  105
other_information_national_agency_to_enforce_solid_waste_laws_and_regulations with null values:  53
other_information_national_law_governing_solid_waste_management_in_the_country with null values:  23
other_information_ppp_rules_and_regulations with null values:  55
other_information_summary_of_key_solid_waste_information_made_available_to_the_public with null value

convertion String to Double

In [99]:
df2 = df2.withColumn("gdp",df2.gdp.cast(DoubleType()))

df2=df2.withColumn('gdp',round(df2.gdp.cast(DoubleType()),2))

In [100]:
col_str = ['gdp','composition_food_organic_waste_percent','composition_glass_percent','composition_metal_percent','composition_other_percent','composition_paper_cardboard_percent','composition_plastic_percent','composition_rubber_leather_percent','composition_wood_percent','composition_yard_garden_green_waste_percent',
           'special_waste_agricultural_waste_tons_year','special_waste_construction_and_demolition_waste_tons_year','special_waste_e_waste_tons_year','special_waste_hazardous_waste_tons_year','special_waste_industrial_waste_tons_year',
           'special_waste_medical_waste_tons_year', 'total_msw_total_msw_generated_tons_year','waste_treatment_recycling_percent', 'waste_treatment_unaccounted_for_percent']
for col in col_str :
    df2 = df2.withColumn(col,df2[col].cast(DoubleType()))
    df2 = df2.withColumn(col,round(df2[col].cast(DoubleType()),2))

In [101]:
df2.printSchema

<bound method DataFrame.printSchema of DataFrame[country_name: string, gdp: double, composition_food_organic_waste_percent: double, composition_glass_percent: double, composition_metal_percent: double, composition_other_percent: double, composition_paper_cardboard_percent: double, composition_plastic_percent: double, composition_rubber_leather_percent: double, composition_wood_percent: double, composition_yard_garden_green_waste_percent: double, other_information_information_system_for_solid_waste_management: string, other_information_national_agency_to_enforce_solid_waste_laws_and_regulations: string, other_information_national_law_governing_solid_waste_management_in_the_country: string, other_information_ppp_rules_and_regulations: string, other_information_summary_of_key_solid_waste_information_made_available_to_the_public: string, population_population_number_of_people: int, special_waste_agricultural_waste_tons_year: double, special_waste_construction_and_demolition_waste_tons_year

In [102]:
df2.head(5)

[Row(country_name='Aruba', gdp=35563.31, composition_food_organic_waste_percent=None, composition_glass_percent=None, composition_metal_percent=None, composition_other_percent=None, composition_paper_cardboard_percent=None, composition_plastic_percent=None, composition_rubber_leather_percent=None, composition_wood_percent=None, composition_yard_garden_green_waste_percent=None, other_information_information_system_for_solid_waste_management=None, other_information_national_agency_to_enforce_solid_waste_laws_and_regulations='Yes', other_information_national_law_governing_solid_waste_management_in_the_country='Yes', other_information_ppp_rules_and_regulations='Yes', other_information_summary_of_key_solid_waste_information_made_available_to_the_public='Yes', population_population_number_of_people=103187, special_waste_agricultural_waste_tons_year=None, special_waste_construction_and_demolition_waste_tons_year=None, special_waste_e_waste_tons_year=None, special_waste_hazardous_waste_tons_ye

Remplacement les valeurs manquantes

In [ ]:
#Find the avg of all numeric columns
from pyspark.sql.functions import avg

def mean_of_pyspark_columns(df, numeric_cols, verbose=False):
    col_with_mean=[]
    for col in numeric_cols:
        mean_value = df.select(avg(df[col]))
        avg_col = mean_value.columns[0]
        res = mean_value.rdd.map(lambda row : row[avg_col]).collect()
        
        if (verbose==True): print(mean_value.columns[0], "\t", res[0])
        col_with_mean.append([col, res[0]])    
    return col_with_mean

In [ ]:
from pyspark.sql.functions import when, lit

def fill_missing_with_mean(df, numeric_cols):
    col_with_mean = mean_of_pyspark_columns(df, numeric_cols) 
    
    for col, mean in col_with_mean:
        df = df.withColumn(col, when(df[col].isNull()==True, 
        lit(mean)).otherwise(df[col]))
        
    return df

In [ ]:
numeric_col = ['gdp','composition_food_organic_waste_percent','composition_glass_percent','composition_metal_percent','composition_other_percent', 'composition_paper_cardboard_percent','composition_plastic_percent','composition_rubber_leather_percent','']

bucket

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import *

Evaluation

Appliquer un algorithme d’apprentissage via Mllib